# Lab 04 · Chunking and Vectorization (Notebook Walkthrough)

**Concept.** The `chunk_vector` profile adds the built-in `SplitSkill` and `AzureOpenAIEmbeddingSkill`. Every chunk now carries a 3072-dim `content_vector`, which unlocks **vector** and **hybrid** retrieval in addition to **full-text**. This lab also shows **scoring profiles** — query-time relevance tuning over the BM25 score.

We ask the **same questions** as the other labs so differences are attributable to the index, not the prompt.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': '<home>\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Ingest with the chunk + vector profile

In [2]:
job = lab.ingest(skill_profile='chunk_vector', reuse=True)
lab.chunk_overview(job)

Reusing existing 'chunk_vector' ingestion (doc_id=52d979ab, 412 chunks). Pass reuse=False to force a fresh run.


{'doc_id': '52d979ab56524d99bc066c7cb4f82db2',
 'skill_profile': 'chunk_vector',
 'chunk_count': 412,
 'avg_tokens': 200.4,
 'max_tokens': 777,
 'chunks_with_summary': 0,
 'chunks_with_keyword_hints': 0,
 'chunks_with_image_description': 0}

## Step 3 — Confirm vectors are present

Vector / hybrid retrieval requires an embedding per chunk. Check that the published chunks were embedded.

In [3]:
from backend.services.indexing import build_foundry_adapter
adapter = build_foundry_adapter()
print('vector search available:', adapter._vector_search_available())

vector search available: True


## Step 4 — Compare retrieval modes on the SAME question

Q1 is precise/keyword-heavy (favours full-text). Q2 is conceptual/multi-hop (favours vector + hybrid).

In [4]:
Q1 = 'What are the objectives of site investigation for ELS works?'
Q2 = 'Given a site with high groundwater and clay layers, what are the key excavation risks and design considerations?'

for mode in ['full_text', 'vector', 'hybrid']:
    print('=' * 80)
    print('MODE:', mode, '| Q2 (conceptual)')
    resp = lab.ask(Q2, job=job, retrieval_mode=mode, record_as=f'lab04_chunkvector_{mode}')
    lab.show_answer(resp, max_citations=4)
    print()

MODE: full_text | Q2 (conceptual)


[retrieval_mode=full_text | scoring_profile=default | citations=8]

However, their locations, length and positioning of response zones should be properly planned with due consideration given to the hydrogeological condition of the groundwater regime adjacent to the site. Where the ground conditions comprise stratified marine clay underlain by sandy soils (e.g. [5]

Supporting evidence:
- Good practice for site investigation and selection of geotechnical parameters that are crucial for the design of ELS works and associated key considerations are presented in Chapter 2. A review of common types of excavation support systems and technical considerations for the design and construction of ELS works are discussed in Chapters 3, 4 and 5. [7]
- Given the temporary nature of ELS works, DGWL should be related to possible scenarios that could occur within the duration of the works for different limit states. It is not necessary to consider the effects of long-term and extreme events (e.g. due t

[retrieval_mode=vector | scoring_profile=default | citations=8]

The presence of loose fill renders a site more susceptible to excessive ground loss during piling operations and bulk excavation, particularly where the groundwater table is near the ground surface. This is commonly the case in reclaimed land where the fill was placed by the hydraulic filling method. [1]

Supporting evidence:
- Site-specific groundwater and drainage conditions should be ascertained within and in the vicinity of the site, and their likely response to, for example, storms, seasonal rise, artesian conditions or tidal action. Field tests normally yield more reliable parameters of the mass permeability of the ground than laboratory tests. [4]
- On the other hand, site supervisory staff should always be alert and take note of any signs of possible ground loss and formation of sinkholes, which typically include the following abnormalities: (a) Larger than expected groundwater discharge seeping into the excavatio

[retrieval_mode=hybrid | scoring_profile=default | citations=8]

Loss of overall stability is likely to occur in excavations near a steeply-sloping site with a high groundwater table, or where a weak subsoil layer (e.g. loose sand or soft clay) is present below the embedded wall. [5]

Supporting evidence:
- In order to minimise disturbance to the adjacent ground due to the boring operation, the pressure of the compressed air should be carefully assessed and monitored, especially for sites with a high groundwater table and thick layers of loose fill, bouldery colluvium or rockfill. Trial boring is usually conducted and should be aimed to determine site-specific  [4]
- Site-specific groundwater and drainage conditions should be ascertained within and in the vicinity of the site, and their likely response to, for example, storms, seasonal rise, artesian conditions or tidal action. ...f any artesian and non-hydrostatic pressures,The marine clay had a low permeability which provided a water

### Side-by-side top hits (Q2)

Look at how the ordering and the winning section change between lexical, semantic, and fused retrieval.

In [5]:
import pandas as pd

rows = []
for mode in ['full_text', 'vector', 'hybrid']:
    for rank, hit in enumerate(lab.retrieve(Q2, job=job, retrieval_mode=mode, top=3), start=1):
        rows.append({'mode': mode, 'rank': rank, 'score': hit['score'], 'section': hit['section'], 'pages': hit['pages']})
pd.DataFrame(rows)

,mode,rank,score,section,pages
0,full_text,1,21.7676,2.2.1 Design for Drained and Undrained Conditions,[21]
1,full_text,2,21.2233,2.1 Site Investigation,[17]
2,full_text,3,19.4262,6.5.2 Design Groundwater Level for Serviceabil...,[71]
3,vector,1,0.7261,5.2 Site Conditions Particularly Vulnerable to...,[53]
4,vector,2,0.7165,5.1 General,[53]
5,vector,3,0.7059,5.7 Site Supervision,"[61, 62]"
6,hybrid,1,0.0317,2.1.2 Groundwater Conditions,"[17, 18]"
7,hybrid,2,0.0141,6.5.1 Design Groundwater Level for Ultimate Li...,[70]
8,hybrid,3,0.0316,2.2.1 Design for Drained and Undrained Conditions,[21]


## Step 5 — Scoring profiles

Scoring profiles re-weight the BM25 score before fusion/semantic ranking. They apply to **full-text** and **hybrid** (not pure vector). Compare the default profile against `enrichment-weighted` (field boosts) on Q1.

In [6]:
import pandas as pd

rows = []
for profile in ['default', 'enrichment-weighted', 'freshness-boosted']:
    for rank, hit in enumerate(lab.retrieve(Q1, job=job, retrieval_mode='full_text', scoring_profile=profile, top=3), start=1):
        rows.append({'scoring_profile': profile, 'rank': rank, 'score': hit['score'], 'section': hit['section']})
pd.DataFrame(rows)

,scoring_profile,rank,score,section
0,default,1,28.9539,2.1 Site Investigation
1,default,2,12.8858,1.2 Overview
2,default,3,12.6903,2.1.1 Ground Investigation
3,enrichment-weighted,1,36.8261,2.1 Site Investigation
4,enrichment-weighted,2,17.6225,2.1.1 Ground Investigation
5,enrichment-weighted,3,14.7221,5.7 Site Supervision
6,freshness-boosted,1,115.8072,2.1 Site Investigation
7,freshness-boosted,2,51.5393,1.2 Overview
8,freshness-boosted,3,50.7576,2.1.1 Ground Investigation


> On the baseline corpus the enrichment fields (`summary_text`, `keyword_hints`) are empty, so `enrichment-weighted` mostly affects results once Lab 05 enrichment is present. `freshness-boosted` uses the indexer `last_updated` high-water mark. The mechanism is identical; the visible effect grows as the index gets richer.

## Takeaways

- Chunking + embeddings unlock **vector** and **hybrid** retrieval over the same document.
- Conceptual/paraphrased questions (Q2) benefit most from vector and hybrid; precise lookups (Q1) stay strong on full-text.
- **Scoring profiles** give you query-time relevance control that is complementary to the semantic ranker.

Next: **Lab 05** adds generative enrichment (summaries + keyword hints) during indexing.